In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
from src.data_processing import (
    load_data,
    calculate_rfm,
    cluster_customers,
    create_high_risk_target,
    merge_high_risk_target
)

In [2]:
df = load_data(
    "../data/raw/data.csv"
)

df.head()

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15T02:18:49Z,2,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15T02:19:08Z,2,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15T02:44:21Z,2,0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15T03:32:55Z,2,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15T03:34:21Z,2,0


In [3]:
rfm = calculate_rfm(df)

rfm.head()

,CustomerId,Recency,Frequency,Monetary
0,CustomerId_1,84,1,-10000.0
1,CustomerId_10,84,1,-10000.0
2,CustomerId_1001,90,5,20000.0
3,CustomerId_1002,26,11,4225.0
4,CustomerId_1003,12,6,20000.0


In [4]:
clustered = cluster_customers(
    rfm
)

clustered.head()

,CustomerId,Recency,Frequency,Monetary,Cluster
0,CustomerId_1,84,1,-10000.0,0
1,CustomerId_10,84,1,-10000.0,0
2,CustomerId_1001,90,5,20000.0,0
3,CustomerId_1002,26,11,4225.0,2
4,CustomerId_1003,12,6,20000.0,2


In [5]:
clustered.groupby(
    "Cluster"
)[
    [
        "Recency",
        "Frequency",
        "Monetary"
    ]
].mean()

,Recency,Frequency,Monetary
Cluster,,,
0,61.859846,7.726699,8.172379e+04
1,29.000000,4091.000000,-1.049000e+08
2,12.716076,34.807692,2.726546e+05


In [6]:
target_df = (
    create_high_risk_target(
        clustered
    )
)

target_df.head()

,CustomerId,Recency,Frequency,Monetary,Cluster,is_high_risk
0,CustomerId_1,84,1,-10000.0,0,1
1,CustomerId_10,84,1,-10000.0,0,1
2,CustomerId_1001,90,5,20000.0,0,1
3,CustomerId_1002,26,11,4225.0,2,0
4,CustomerId_1003,12,6,20000.0,2,0


In [7]:
target_df[
    "is_high_risk"
].value_counts()

is_high_risk
0    2315
1    1427
Name: count, dtype: int64

In [8]:
final_df = (
    merge_high_risk_target(df)
)

final_df.head()

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult,is_high_risk
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15T02:18:49Z,2,0,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15T02:19:08Z,2,0,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15T02:44:21Z,2,0,1
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15T03:32:55Z,2,0,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15T03:34:21Z,2,0,0


In [17]:
final_df = merge_high_risk_target(df)

final_df.to_csv(
    "../data/processed/processed_data.csv",
    index=False
)

In [22]:
model_df = (
    final_df
    .groupby("CustomerId")
    .agg(
        TotalTransactionAmount=(
            "Amount",
            "sum"
        ),
        AverageTransactionAmount=(
            "Amount",
            "mean"
        ),
        TransactionCount=(
            "Amount",
            "count"
        ),
        StdTransactionAmount=(
            "Amount",
            "std"
        ),
        FraudResult=(
            "FraudResult",
            "mean"
        ),
        is_high_risk=(
            "is_high_risk",
            "first"
        )
    )
    .reset_index()
)

In [23]:
model_df.shape

(3742, 7)

In [24]:
model_df.to_csv(
    "../data/processed/model_data.csv",
    index=False
)